# CAVAanalytics: country-scale climate analysis

This practical follows one question: **How could extreme heat change in Kenya?**

We will load a multi-model dataset, examine observations and model biases, calculate future indicators, map the climate change signal and export a raster.

## 1. Start the R kernel

Open this notebook with **R (CAVA)**. The remote-data cell is the slowest step and requires internet access.

In [ ]:
library(CAVAanalytics)
library(ggplot2)
library(patchwork)

packageVersion("CAVAanalytics")

getwd()


[1] ‘4.0.7’

## 2. Define the analysis

The training uses the pre-bias-corrected CORDEX-CORE archive. Keep `bias.correction = FALSE` in later calls.

In [ ]:
COUNTRY <- "Kenya"
DOMAIN <- "AFR-22"
DATASET <- "CORDEX-CORE-BC"
VARIABLE <- "tasmax"

BASELINE_YEARS <- 1991:2000
FUTURE_YEARS <- 2041:2050
HEAT_THRESHOLD_C <- 35
SEASONS <- list(1:3, 4:6, 7:9, 10:12)

dir.create("data")
dir.create("outputs")

## 3. Load observations, historical simulations and projections

`load_data()` selects Kenya, resolves the remote files, streams the required grid cells and harmonizes units and time.

In [ ]:
cache_file <- file.path("data", "kenya_tasmax_1991_2000_2041_2050.rds")

if (file.exists(cache_file)) {
  kenya_tasmax <- readRDS(cache_file)
} else {
  kenya_tasmax <- load_data(
    path.to.data = DATASET,
    path.to.obs = "ERA5",
    country = COUNTRY,
    variable = VARIABLE,
    years.hist = BASELINE_YEARS,
    years.obs = BASELINE_YEARS,
    years.proj = FUTURE_YEARS,
    domain = DOMAIN,
    n.sessions = 4
  )
  saveRDS(kenya_tasmax, cache_file)
}

## 4. Inspect the loaded object

In [ ]:
class(kenya_tasmax)
names(attributes(kenya_tasmax))
colnames(kenya_tasmax[[1]])
kenya_tasmax[[1]]$experiment
dim(kenya_tasmax[[1]]$models_mbrs[[1]]$Data)

## 5. Historical climate and a threshold indicator

First map the mean observed maximum temperature. Then count the annual number of days above 35 °C by season.

In [ ]:
obs_mean <- observations(
  kenya_tasmax,
  season = list(1:12)
)

plotting(
  obs_mean,
  plot_titles = "Mean maximum temperature (°C)",
  palette = IPCC_palette(type = "tmp", divergent = FALSE)
)

In [ ]:
obs_hot_days <- observations(
  kenya_tasmax,
  season = SEASONS,
  uppert = HEAT_THRESHOLD_C
)

plotting(
  obs_hot_days,
  plot_titles = "Days above 35 °C",
  palette = IPCC_palette(type = "tmp", divergent = FALSE)
)

## 6. Model biases

Compare historical model simulations with ERA5. These data already received bias correction, so the function evaluates the remaining bias without applying another correction.

In [ ]:
heat_bias <- model_biases(
  kenya_tasmax,
  season = list(1:12),
  uppert = HEAT_THRESHOLD_C,
  bias.correction = FALSE
)

plotting(
  heat_bias,
  ensemble = TRUE,
  plot_titles = "Bias in days above 35 °C",
  palette = IPCC_palette(type = "tmp", divergent = TRUE)
)

## 7. Future projections

`projections()` can return the ensemble mean, ensemble spread, individual models and annual time series.

In [ ]:
future_hot_days <- projections(
  kenya_tasmax,
  season = SEASONS,
  uppert = HEAT_THRESHOLD_C,
  bias.correction = FALSE
)

p_mean <- plotting(
  future_hot_days,
  ensemble = TRUE,
  stat = "mean",
  plot_titles = "Projected days above 35 °C",
  palette = IPCC_palette(type = "tmp", divergent = FALSE)
)

p_spread <- plotting(
  future_hot_days,
  ensemble = TRUE,
  stat = "sd",
  plot_titles = "Ensemble standard deviation"
)

p_mean / p_spread

In [ ]:
plotting(
  future_hot_days,
  ensemble = TRUE,
  temporal = TRUE,
  plot_titles = "Annual days above 35 °C",
  palette = c("#0072B2", "#D55E00")
)

## 8. Consecutive-day indicators

The same analysis functions can calculate the longest hot spell or the number of heat waves.

In [ ]:
longest_hot_spell <- projections(
  kenya_tasmax,
  season = list(1:12),
  uppert = HEAT_THRESHOLD_C,
  consecutive = TRUE,
  duration = "max",
  bias.correction = FALSE
)

plotting(
  longest_hot_spell,
  ensemble = TRUE,
  plot_titles = "Longest spell above 35 °C",
  palette = IPCC_palette(type = "tmp", divergent = FALSE)
)

In [ ]:
heatwave_frequency <- projections(
  kenya_tasmax,
  season = list(1:12),
  uppert = HEAT_THRESHOLD_C,
  consecutive = TRUE,
  duration = 3,
  frequency = TRUE,
  bias.correction = FALSE
)

plotting(
  heatwave_frequency,
  ensemble = TRUE,
  plot_titles = "Heat waves per year",
  palette = IPCC_palette(type = "tmp", divergent = FALSE)
)

## 9. Climate change signal and model agreement

Compare the future and historical periods. Stippling follows the package convention for insufficient agreement on the sign of change.

In [ ]:
heat_change <- climate_change_signal(
  kenya_tasmax,
  season = SEASONS,
  uppert = HEAT_THRESHOLD_C,
  threshold = 0.80,
  bias.correction = FALSE
)

plotting(
  heat_change,
  ensemble = TRUE,
  plot_titles = "Change in days above 35 °C",
  palette = IPCC_palette(type = "tmp", divergent = TRUE),
  point_size = 0.08
)

In [ ]:
plotting(
  heat_change,
  ensemble = TRUE,
  temporal = TRUE,
  plot_titles = "Annual change in hot days",
  palette = c("#0072B2", "#D55E00")
)

## 10. Export the analytical result

The export writes ensemble mean and standard-deviation GeoTIFFs to `outputs/`.

In [ ]:
extract_raster(
  heat_change,
  file.extension = "kenya_heat_change.tif",
  ensemble = TRUE,
  path = "outputs/"
)

list.files("outputs", pattern = "\\.tif$", full.names = TRUE)

## Optional extensions

- Repeat the workflow with `variable = "pr"` and `aggr.m = "sum"`.
- Use `percentage = TRUE` in `climate_change_signal()` for relative precipitation change.
- Use `load_data_and_projections()` or `load_data_and_climate_change_signal()` for large regions processed in spatial chunks.
- Use a custom area with `country = NULL`, `xlim` and `ylim`.

Documentation: [CAVAanalytics introduction](https://un-fao.github.io/CAVAanalytics/articles/Introduction.html) and [advanced indicators](https://un-fao.github.io/CAVAanalytics/articles/more_advanced.html).

## Takeaway

CAVAanalytics connects remote climate data to country-scale indicators, ensemble interpretation, maps and GIS-ready outputs in one reproducible R workflow.